In [1]:
import os, json, contextlib
from pathlib import Path
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from tqdm import tqdm
import pandas as pd
from scipy.ndimage import affine_transform
from scipy.ndimage import label as cc_label
from scipy.stats import spearmanr
from medpy.metric.binary import hd95 as medpy_hd95
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from nnunet_mednext.network_architecture.mednextv1.create_mednext_v1 import create_mednext_v1

SPLITS_DIR = Path(r"D:\master_experiments\data\splits\BraTS2020_Splits")
META_PATH  = SPLITS_DIR / "splits_metadata.json"

MEDNEXT_BASE = Path(r"D:\master_experiments\models_configs\MedNeXt_BraTS2020")
MEDNEXT_CKPT = MEDNEXT_BASE / "checkpoints"
MEDNEXT_PRED = MEDNEXT_BASE / "predictions_test"
MEDNEXT_LOGS = MEDNEXT_BASE / "logs"

for p in [MEDNEXT_BASE, MEDNEXT_CKPT, MEDNEXT_PRED, MEDNEXT_LOGS]:
    p.mkdir(parents=True, exist_ok=True)


MODS      = ["flair", "t1", "t1ce", "t2"]
N_CLASSES = 4  
PATCH     = 128
NUM_ITERS_PER_EPOCH = 250

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()
NW      = 0 if os.name == "nt" else 4

print("Device:", DEVICE)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | VRAM: {props.total_memory/1e9:.1f} GB")

Device: cuda
GPU: NVIDIA GeForce RTX 4070 SUPER | VRAM: 12.9 GB


In [2]:
with open(META_PATH, "r", encoding="utf-8") as f:
    meta = json.load(f)

train_ids = meta["ids"]["train"]
val_ids   = meta["ids"]["val"]
test_ids  = meta["ids"]["test"]

train_set, val_set, test_set = set(train_ids), set(val_ids), set(test_ids)
assert len(train_set & val_set) == 0
assert len(train_set & test_set) == 0
assert len(val_set & test_set) == 0

print("Counts:", len(train_ids), len(val_ids), len(test_ids))
print("OK: sem repetição entre splits")

Counts: 245 52 53
OK: sem repetição entre splits


In [3]:
# Funções auxiliares

def case_dir(split_name: str, case_id: str) -> Path:
    return SPLITS_DIR / split_name / case_id

def find_file(folder: Path, key: str) -> Path:
    for cand in [folder / f"{key}.nii.gz", folder / f"{key}.nii"]:
        if cand.exists():
            return cand
    cands = sorted(list(folder.glob(f"*{key}*.nii*")))
    if key == "t1":
        cands = [c for c in cands if "t1ce" not in c.name.lower()]
    if not cands:
        raise FileNotFoundError(f"{key} not found in {folder}")
    return cands[0]

def load_arr(p) -> np.ndarray:
    return np.squeeze(np.asanyarray(nib.load(str(p)).dataobj))

def norm_zscore_fg(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32)
    mask = x > 0
    if mask.sum() == 0:
        return np.zeros_like(x)
    mean = float(x[mask].mean())
    std  = float(x[mask].std()) + 1e-8
    out  = (x - mean) / std
    out[~mask] = 0.0
    return out

def norm01(x, p1=1, p99=99) -> np.ndarray:
    x = x.astype(np.float32)
    lo, hi = np.percentile(x, [p1, p99])
    if hi <= lo:
        return np.zeros_like(x)
    return np.clip((x - lo) / (hi - lo), 0, 1)

def load_brats_seg(path) -> np.ndarray:
    data = load_arr(path).astype(np.int16)
    data[data == 4] = 3
    return data

def dice_score(a, b) -> float:
    inter = np.count_nonzero(a & b)
    denom = np.count_nonzero(a) + np.count_nonzero(b)
    return 1.0 if denom == 0 else (2.0 * inter / denom)

def pick_best_slice(seg, axis=2) -> int:
    counts = np.sum(seg > 0, axis=tuple(i for i in range(3) if i != axis))
    return int(np.argmax(counts))

for cid in train_ids[:3]:
    d = case_dir("train", cid)
    assert d.exists()
    _ = [find_file(d, m) for m in MODS]
    _ = find_file(d, "seg")
print("Helpers OK")

Helpers OK


In [4]:
class BraTSPatchDataset(Dataset):

    def __init__(self, ids_list, split_name, patch=PATCH,
                 augment=True, num_iterations=None):
        self.ids            = ids_list
        self.split          = split_name
        self.patch          = patch
        self.aug            = augment
        self.num_iterations = num_iterations 

    def __len__(self):
        return self.num_iterations if self.num_iterations is not None else len(self.ids)

    def _lesion_crop(self, arrays, patch):
        seg  = arrays[-1]
        half = patch // 2
        if np.random.rand() < 0.33:
            center = np.array([
                np.random.randint(half, max(dim - half, half + 1))
                for dim in seg.shape
            ])
        else:
            coords = np.argwhere(seg > 0)
            if len(coords) == 0:
                center = np.array([
                    np.random.randint(half, max(dim - half, half + 1))
                    for dim in seg.shape
                ])
            else:
                center = coords[np.random.randint(len(coords))]
        lo, hi = [], []
        for ax in range(3):
            c = int(np.clip(center[ax], half, seg.shape[ax] - half))
            lo.append(c - half)
            hi.append(c + half)
        return [a[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]] for a in arrays]

    def _augment(self, imgs, seg):
        if np.random.rand() < 0.2:
            angle  = np.random.uniform(-30, 30) * np.pi / 180.0
            scale  = np.random.uniform(0.85, 1.25)
            cos_a  = np.cos(angle) / scale
            sin_a  = np.sin(angle) / scale
            matrix = np.array([[1.0/scale, 0., 0.],
                                [0., cos_a, sin_a],
                                [0., -sin_a, cos_a]], dtype=np.float64)
            shape  = np.array(imgs[0].shape)
            center = (shape - 1) / 2.0
            offset = center - matrix @ center
            imgs = [affine_transform(im, matrix, offset=offset, order=1,
                                     mode='constant', cval=0).astype(np.float32)
                    for im in imgs]
            seg  = affine_transform(seg.astype(np.int16), matrix, offset=offset,
                                    order=0, mode='constant', cval=0).astype(np.int16)
        for ax in range(3):
            if np.random.rand() > 0.5:
                imgs = [np.flip(im, axis=ax).copy() for im in imgs]
                seg  = np.flip(seg,  axis=ax).copy()
        bright = np.random.uniform(0.85, 1.15) if np.random.rand() > 0.5 else 1.0
        aug_imgs = []
        for im in imgs:
            im = im * bright
            if np.random.rand() > 0.5:
                im = im + np.random.normal(0, 0.05, im.shape).astype(np.float32)
            aug_imgs.append(im.astype(np.float32))
        return aug_imgs, seg

    def __getitem__(self, idx):
        cid = (self.ids[np.random.randint(len(self.ids))]
               if self.num_iterations is not None else self.ids[idx])
        d   = case_dir(self.split, cid)
        imgs = [norm_zscore_fg(load_arr(find_file(d, m))) for m in MODS]
        seg  = load_brats_seg(find_file(d, "seg"))
        cropped      = self._lesion_crop(imgs + [seg], self.patch)
        imgs, seg    = cropped[:-1], cropped[-1]
        if self.aug:
            imgs, seg = self._augment(imgs, seg)
        vol   = torch.from_numpy(np.stack(imgs, axis=0).astype(np.float32))
        seg_t = torch.from_numpy(seg.astype(np.int64))
        return vol, seg_t

In [5]:
# ═══════════════════════════════════════════════════════════
#  Instanciação do MedNeXt via create_mednext_v1 (API oficial)
# ═══════════════════════════════════════════════════════════
#
#  model_id     = 'S'  → Small  (~5.6 M params, exp_r=2, block_counts=[2,2,2,2,2,2,2,2,2])
#                 'B'  → Base   (~10 M params)
#                 'M'  → Medium (~18 M params, com gradient checkpointing)
#                 'L'  → Large  (~63 M params, com gradient checkpointing)
#
#  kernel_size  = 3    → kernel depthwise 3×3×3
#                 5    → kernel 5×5×5 (maior receptive field, mais VRAM)
#
#  num_input_channels = 4  → FLAIR, T1, T1ce, T2
#  num_classes        = 4  → {0: bg, 1: necrose/non-enh, 2: edema, 3: ET}
#  deep_supervision   = True → saídas auxiliares nos decoders (pesos decrescentes)
# ═══════════════════════════════════════════════════════════

MEDNEXT_MODEL_ID   = 'S'   # 'S' | 'B' | 'M' | 'L'
MEDNEXT_KERNEL     = 3     # 3 | 5

model = create_mednext_v1(
    num_input_channels = 4,
    num_classes        = N_CLASSES,
    model_id           = MEDNEXT_MODEL_ID,
    kernel_size        = MEDNEXT_KERNEL,
    deep_supervision   = True,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6

print("=" * 55)
print(f"  MedNeXt-{MEDNEXT_MODEL_ID} (k{MEDNEXT_KERNEL})  —  Configuração do modelo")
print("=" * 55)
print(f"  Canais de entrada (in_channels) : {4}")
print(f"  Canais de saída  (n_classes)    : {N_CLASSES}")
print(f"  Device                          : {DEVICE}")
print(f"  Parâmetros treináveis           : {n_params:.2f} M")
print("=" * 55)

  MedNeXt-S (k3)  —  Configuração do modelo
  Canais de entrada (in_channels) : 4
  Canais de saída  (n_classes)    : 4
  Device                          : cuda
  Parâmetros treináveis           : 5.55 M


In [6]:
class DeepSupervisionLoss(nn.Module):

    def __init__(self, n_levels: int = 4):
        super().__init__()
        raw  = [1.0 / (2 ** l) for l in range(n_levels)]
        s    = sum(raw)
        self.weights  = [w / s for w in raw]
        self.n_levels = n_levels

    def _soft_dice(self, pred_soft, target_oh):
        smooth = 1e-5
        dims   = (0, 2, 3, 4)
        p = pred_soft[:, 1:]  
        t = target_oh[:, 1:]
        inter = (p * t).sum(dims)
        denom = p.sum(dims) + t.sum(dims)
        return 1.0 - ((2.0 * inter + smooth) / (denom + smooth)).mean()

    def forward(self, outputs, target):
        if not isinstance(outputs, (list, tuple)):
            outputs = [outputs]
        n = min(len(outputs), self.n_levels)
        total = 0.0
        for lvl in range(n):
            logits = outputs[lvl]
            w      = self.weights[lvl]
            if lvl == 0:
                tgt = target
            else:
                scale = 1.0 / (2 ** lvl)
                tgt = F.interpolate(
                    target.float().unsqueeze(1),
                    scale_factor=scale, mode="nearest"
                ).squeeze(1).long()
            n_cls = logits.shape[1]
            oh    = F.one_hot(tgt, n_cls).permute(0, 4, 1, 2, 3).float()
            soft  = torch.softmax(logits, dim=1)
            total += w * (self._soft_dice(soft, oh) + F.cross_entropy(logits, tgt))
        return total

In [7]:
TRAIN_EPOCHS = 100
BATCH_SIZE   = 1
ACCUM_STEPS  = 4
LR           = 1e-3
GRAD_CLIP    = 1.0


def train_mednext(model, train_ids, val_ids,
                  epochs=TRAIN_EPOCHS, save_dir=MEDNEXT_CKPT):

    crit   = DeepSupervisionLoss(n_levels=4)
    scaler = GradScaler("cuda") if USE_AMP else None

    ds_tr = BraTSPatchDataset(train_ids, "train", augment=True,
                              num_iterations=NUM_ITERS_PER_EPOCH)
    ds_va = BraTSPatchDataset(val_ids,   "val",   augment=False)
    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NW, pin_memory=USE_AMP)
    dl_va = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False,
                       num_workers=NW, pin_memory=USE_AMP)

    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)

    tr_losses, va_losses = [], []
    best_val = float("inf")

    for epoch in range(epochs):
        model.train()
        running = 0.0
        opt.zero_grad()

        for step, (vol, seg) in enumerate(
                tqdm(dl_tr, desc=f"Ep {epoch+1:3d}/{epochs} train", leave=False)):
            vol = vol.to(DEVICE)
            seg = seg.to(DEVICE)

            with autocast("cuda") if USE_AMP else contextlib.nullcontext():
                out  = model(vol)
                loss = crit(out, seg) / ACCUM_STEPS

            if scaler:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            if (step + 1) % ACCUM_STEPS == 0:
                if scaler:
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                    scaler.step(opt); scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                    opt.step()
                opt.zero_grad()

            running += loss.item() * ACCUM_STEPS

        if len(dl_tr) % ACCUM_STEPS != 0:
            if scaler:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(opt); scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                opt.step()
            opt.zero_grad()

        avg_tr = running / max(len(dl_tr), 1)
        tr_losses.append(avg_tr)
        sched.step()

        model.eval()
        running_v = 0.0
        with torch.no_grad():
            for vol, seg in dl_va:
                vol = vol.to(DEVICE); seg = seg.to(DEVICE)
                with autocast("cuda") if USE_AMP else contextlib.nullcontext():
                    running_v += crit(model(vol), seg).item()

        avg_va = running_v / max(len(dl_va), 1)
        va_losses.append(avg_va)
        print(f"  Ep {epoch+1:3d} | tr={avg_tr:.4f} | va={avg_va:.4f}")

        if avg_va < best_val:
            best_val = avg_va
            torch.save(model.state_dict(), save_dir / "mednext_best.pth")
            print(f"    ✓ Melhor checkpoint salvo (val={best_val:.4f})")

    history = {"train": tr_losses, "val": va_losses}
    with open(MEDNEXT_LOGS / "train_history.json", "w") as f:
        json.dump(history, f, indent=2)
    print("\nTreino concluído.")
    return history


train_history = train_mednext(model, train_ids, val_ids)

KeyboardInterrupt: 

In [ ]:
def _gaussian_kernel_3d(size: int) -> np.ndarray:
    sigma = size / 8
    ax    = np.arange(size) - size // 2
    g     = np.exp(-0.5 * (ax / sigma) ** 2)
    g3d   = g[:, None, None] * g[None, :, None] * g[None, None, :]
    return (g3d / g3d.max()).astype(np.float32)


def sliding_window_inference(model, vol_4ch: np.ndarray, n_classes: int,
                              patch: int = PATCH, overlap: float = 0.5) -> np.ndarray:
    model.eval()
    _, D, H, W = vol_4ch.shape
    step       = max(int(patch * (1.0 - overlap)), 1)

    accum   = np.zeros((n_classes, D, H, W), dtype=np.float32)
    weights = np.zeros((D, H, W),            dtype=np.float32)
    gauss   = _gaussian_kernel_3d(patch)

    def _make_range(dim):
        last = max(dim - patch, 0)
        r    = list(range(0, last + 1, step))
        if r and r[-1] != last:
            r.append(last)
        return r if r else [0]

    with torch.no_grad():
        for z0 in _make_range(D):
            for y0 in _make_range(H):
                for x0 in _make_range(W):
                    z1 = min(z0 + patch, D); z0_ = z1 - patch
                    y1 = min(y0 + patch, H); y0_ = y1 - patch
                    x1 = min(x0 + patch, W); x0_ = x1 - patch

                    t = torch.from_numpy(
                        vol_4ch[:, z0_:z1, y0_:y1, x0_:x1][None]).to(DEVICE)

                    with autocast("cuda") if USE_AMP else contextlib.nullcontext():
                        out = model(t)
                        # MedNeXt retorna lista com deep_supervision=True
                        if isinstance(out, (list, tuple)):
                            out = out[0]
                        prob = torch.softmax(out, dim=1)[0].cpu().float().numpy()

                    accum[:, z0_:z1, y0_:y1, x0_:x1] += prob * gauss[None]
                    weights[z0_:z1, y0_:y1, x0_:x1]  += gauss

    accum /= np.maximum(weights[None], 1e-8)
    return accum.argmax(axis=0).astype(np.int16)


print("sliding_window_inference OK")

In [ ]:
# Carrega melhor checkpoint e gera predições no conjunto de teste
model.load_state_dict(
    torch.load(MEDNEXT_CKPT / "mednext_best.pth", map_location=DEVICE,
               weights_only=True))
print("Melhor modelo MedNeXt carregado.")

model.eval()
print(f"\nGerando predições → {MEDNEXT_PRED}")

for cid in tqdm(test_ids, desc="Predição test"):
    out_path = MEDNEXT_PRED / f"{cid}.nii.gz"
    d   = case_dir("test", cid)
    ref = nib.load(str(find_file(d, "flair")))
    imgs = [norm_zscore_fg(load_arr(find_file(d, m))) for m in MODS]
    vol  = np.stack(imgs, axis=0).astype(np.float32)  # [4, D, H, W]

    pred = sliding_window_inference(model, vol, n_classes=N_CLASSES)
    nib.save(nib.Nifti1Image(pred, ref.affine, ref.header), str(out_path))

print("Predições concluídas.")

In [ ]:
# Dice global por caso 

def get_pred_path(cid: str) -> Path:
    for ext in [".nii.gz", ".nii"]:
        p = MEDNEXT_PRED / f"{cid}{ext}"
        if p.exists():
            return p
    raise FileNotFoundError(f"Pred não encontrada para {cid}")

def dice(a, b) -> float:
    inter = np.count_nonzero(a & b)
    denom = np.count_nonzero(a) + np.count_nonzero(b)
    return 1.0 if denom == 0 else (2.0 * inter / denom)

rows = []
for cid in tqdm(test_ids, desc="Dice por caso"):
    gt = load_brats_seg(find_file(case_dir("test", cid), "seg"))
    pr = load_arr(get_pred_path(cid)).astype(np.int16)

    d1 = dice(gt == 1, pr == 1)
    d2 = dice(gt == 2, pr == 2)
    d3 = dice(gt == 3, pr == 3)
    wt = dice(((gt==1)|(gt==2)|(gt==3)), ((pr==1)|(pr==2)|(pr==3)))
    tc = dice(((gt==1)|(gt==3)),         ((pr==1)|(pr==3)))

    rows.append({"id": cid,
                 "dice_c1": d1, "dice_c2": d2, "dice_ET": d3,
                 "dice_WT": wt, "dice_TC": tc})

df = pd.DataFrame(rows)
df.to_csv(MEDNEXT_LOGS / "dice_test.csv", index=False)
df.describe()

In [ ]:
# HD95 (Hausdorff Distance 95th percentile) por caso — mesmo padrão do Dice global, por regiões

def hd95_score(gt_mask, pr_mask, voxelspacing=None):
    gt_b = gt_mask.astype(bool)
    pr_b = pr_mask.astype(bool)
    if not gt_b.any() and not pr_b.any():
        return 0.0
    if not gt_b.any() or not pr_b.any():
        return np.nan
    return float(medpy_hd95(pr_b, gt_b, voxelspacing=voxelspacing))


hd95_rows = []
for cid in tqdm(test_ids, desc="HD95 por caso"):
    gt_path = find_file(case_dir("test", cid), "seg")
    pr_path = get_pred_path(cid)

    gt = load_brats_seg(gt_path)
    pr = load_arr(pr_path).astype(np.int16)

    spacing = nib.load(str(gt_path)).header.get_zooms()[:3]

    h1  = hd95_score(gt == 1, pr == 1, voxelspacing=spacing)
    h2  = hd95_score(gt == 2, pr == 2, voxelspacing=spacing)
    h3  = hd95_score(gt == 3, pr == 3, voxelspacing=spacing)
    hwt = hd95_score(((gt==1)|(gt==2)|(gt==3)), ((pr==1)|(pr==2)|(pr==3)), voxelspacing=spacing)
    htc = hd95_score(((gt == 1) | (gt == 3)),   ((pr == 1) | (pr == 3)),   voxelspacing=spacing)

    hd95_rows.append({
        "id":       cid,
        "hd95_c1":  h1,
        "hd95_c2":  h2,
        "hd95_ET":  h3,
        "hd95_WT":  hwt,
        "hd95_TC":  htc,
    })

df_hd95 = pd.DataFrame(hd95_rows)
df_hd95.to_csv(MEDNEXT_LOGS / "hd95_test.csv", index=False)
df_hd95.describe()

In [ ]:
# Análise condicional 1/3: frequência de presença de cada classe no GT

presence_rows = []
for cid in tqdm(test_ids, desc="Presença GT"):
    gt = load_brats_seg(find_file(case_dir("test", cid), "seg"))
    presence_rows.append({
        "id":     cid,
        "has_c1": bool((gt == 1).any()),
        "has_c2": bool((gt == 2).any()),
        "has_ET": bool((gt == 3).any()),
        "has_WT": bool(((gt==1)|(gt==2)|(gt==3)).any()),
        "has_TC": bool(((gt==1)|(gt==3)).any()),
    })
pres_df = pd.DataFrame(presence_rows)
df_full = df.merge(pres_df, on="id")

df_presence = pd.DataFrame([
    {"classe":     col,
     "n_presente": int(df_full[col].sum()),
     "n_total":    len(df_full),
     "pct":        100.0 * df_full[col].sum() / len(df_full)}
    for col in ["has_c1", "has_c2", "has_ET", "has_WT", "has_TC"]
])
df_presence.round(2)

In [ ]:
# Análise condicional 2/3: Dice somente nos casos em que a classe está presente no GT
pairs = [("dice_c1","has_c1"), ("dice_c2","has_c2"), ("dice_ET","has_ET"),
         ("dice_WT","has_WT"), ("dice_TC","has_TC")]
df_conditional = pd.DataFrame({
    col: df_full.loc[df_full[has], col].reset_index(drop=True)
    for col, has in pairs
})
df_conditional.to_csv(MEDNEXT_LOGS / "dice_conditional.csv", index=False)
print("\nDice condicional:")
df_conditional.describe().round(4)

In [ ]:
# Análise condicional 3/3: bimodalidade — % de Dice=0 e % de Dice≈1

df_bimodality = pd.DataFrame([
    {"metric":         col,
     "pct_dice_zero": float((df_full[col] == 0.0).mean()) * 100,
     "pct_dice_one":  float((df_full[col] >= 0.999).mean()) * 100}
    for col, _ in pairs
])
df_bimodality.to_csv(MEDNEXT_LOGS / "dice_bimodality.csv", index=False)
df_bimodality.round(2)

In [ ]:
# Lesion-wise Dice (LWD) — métrica primária do BraTS 2023+

LW_MIN_SIZE = 50

def lesion_wise_dice(gt_mask: np.ndarray, pr_mask: np.ndarray,
                     min_size: int = LW_MIN_SIZE) -> float:
    structure = np.ones((3, 3, 3), dtype=int)
    if not gt_mask.any() and not pr_mask.any():
        return 1.0
    gt_lab, n_gt = cc_label(gt_mask.astype(bool), structure=structure)
    pr_lab, n_pr = cc_label(pr_mask.astype(bool), structure=structure)
    dice_list, matched_pred = [], set()
    for g in range(1, n_gt + 1):
        gt_l = (gt_lab == g)
        if gt_l.sum() < min_size:
            continue
        overlap_ids = np.unique(pr_lab[gt_l])
        overlap_ids = overlap_ids[overlap_ids > 0]
        if len(overlap_ids) == 0:
            dice_list.append(0.0)  # false negative
        else:
            pr_match = np.isin(pr_lab, overlap_ids)
            inter = np.logical_and(gt_l, pr_match).sum()
            denom = gt_l.sum() + pr_match.sum()
            dice_list.append(2.0 * inter / denom if denom > 0 else 0.0)
            matched_pred.update(int(p) for p in overlap_ids)
    for p in range(1, n_pr + 1):
        if p in matched_pred:
            continue
        if (pr_lab == p).sum() < min_size:
            continue
        dice_list.append(0.0)  # false positive
    return float(np.mean(dice_list)) if dice_list else 1.0


lw_rows = []
for cid in tqdm(test_ids, desc="Lesion-wise Dice"):
    gt = load_brats_seg(find_file(case_dir("test", cid), "seg"))
    pr = load_arr(get_pred_path(cid)).astype(np.int16)

    gt_wt = (gt==1)|(gt==2)|(gt==3);  pr_wt = (pr==1)|(pr==2)|(pr==3)
    gt_tc = (gt==1)|(gt==3);          pr_tc = (pr==1)|(pr==3)
    gt_et = (gt==3);                  pr_et = (pr==3)

    lw_rows.append({"id":     cid,
                    "lwd_WT": lesion_wise_dice(gt_wt, pr_wt),
                    "lwd_TC": lesion_wise_dice(gt_tc, pr_tc),
                    "lwd_ET": lesion_wise_dice(gt_et, pr_et)})

lw_df = pd.DataFrame(lw_rows)
lw_df.to_csv(MEDNEXT_LOGS / "dice_lesionwise.csv", index=False)
lw_df[["lwd_WT", "lwd_TC", "lwd_ET"]].describe().round(4)

In [ ]:
# Curvas de loss (treino vs validação)

hist_path = MEDNEXT_LOGS / "train_history.json"
if "train_history" not in vars():
    if not hist_path.exists():
        raise FileNotFoundError(
            f"Execute a célula de treino primeiro, ou verifique: {hist_path}")
    with open(hist_path) as f:
        train_history = json.load(f)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_history["train"], label="train")
ax.plot(train_history["val"],   label="val", ls="--")
ax.set_xlabel("Época")
ax.set_ylabel("Loss (Dice+CE, deep sup.)")
ax.set_title(f"MedNeXt-{MEDNEXT_MODEL_ID} (k{MEDNEXT_KERNEL}) BraTS2020 — {TRAIN_EPOCHS} épocas")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(MEDNEXT_LOGS / "loss_curves.png", dpi=150)
plt.show()

In [ ]:
cmap_mask = ListedColormap([(0,0,0,1), (1,0,0,1), (0,1,0,1), (0,0,1,1)])
norm_mask = BoundaryNorm([0, 1, 2, 3, 4], cmap_mask.N)

def masks_from_seg(seg):
    nec = seg == 1; ede = seg == 2; enh = seg == 3
    wt  = (seg==1)|(seg==2)|(seg==3); tc  = (seg==1)|(seg==3)
    return nec, ede, enh, wt, tc

def overlay(ax, base2d, mask2d, color_rgb, alpha=0.45, title=""):
    ax.imshow(base2d, cmap="gray", origin="lower")
    rgba = np.zeros((*mask2d.shape, 4), dtype=np.float32)
    rgba[...,0], rgba[...,1], rgba[...,2] = color_rgb
    rgba[...,3] = mask2d.astype(np.float32) * alpha
    ax.imshow(rgba, origin="lower")
    ax.set_title(title, fontsize=8); ax.axis("off")

def plot_random_case_multimodal_gt_pred(
        ids_list, split_name="test", seed=None, z=None,
        alpha_cls=0.45, alpha_comp=0.35):
    rng = np.random.default_rng(seed)
    cid = str(rng.choice(ids_list))
    d   = case_dir(split_name, cid)

    gt = load_brats_seg(find_file(d, "seg"))
    pr = load_arr(get_pred_path(cid)).astype(np.int16)

    if z is None:
        z = pick_best_slice(gt)

    d1 = dice(gt==1, pr==1); d2 = dice(gt==2, pr==2); d3 = dice(gt==3, pr==3)
    wt = dice(((gt==1)|(gt==2)|(gt==3)), ((pr==1)|(pr==2)|(pr==3)))
    tc = dice(((gt==1)|(gt==3)),         ((pr==1)|(pr==3)))

    print(f"\nPaciente: {cid} | split={split_name} | z={z}")
    print(f"Dice C1 (necrose/non-enh): {d1:.4f}")
    print(f"Dice C2 (edema):           {d2:.4f}")
    print(f"Dice ET (enhancing):       {d3:.4f}")
    print(f"Dice WT:                   {wt:.4f}")
    print(f"Dice TC:                   {tc:.4f}\n")

    gt2d = gt[:,:,z].T; pr2d = pr[:,:,z].T
    gt_nec,gt_ede,gt_enh,gt_wt,gt_tc = masks_from_seg(gt2d)
    pr_nec,pr_ede,pr_enh,pr_wt,pr_tc = masks_from_seg(pr2d)

    fig, axes = plt.subplots(2*len(MODS), 7, figsize=(28, 3.2*2*len(MODS)))
    fig.suptitle(f"MedNeXt | {cid} | z={z}", y=1.01)

    for i, mod in enumerate(MODS):
        img2d = norm01(load_arr(find_file(d, mod))[:,:,z]).T
        r_gt, r_pr = 2*i, 2*i+1

        axes[r_gt,0].imshow(img2d, cmap="gray", origin="lower")
        axes[r_gt,0].set_title(f"GT • {mod.upper()}", fontsize=8); axes[r_gt,0].axis("off")
        axes[r_gt,1].imshow(gt2d, cmap=cmap_mask, norm=norm_mask, origin="lower")
        axes[r_gt,1].set_title("GT • Mask", fontsize=8); axes[r_gt,1].axis("off")
        overlay(axes[r_gt,2], img2d, gt_ede, (0,1,0), alpha_cls, "GT • Edema (2)")
        overlay(axes[r_gt,3], img2d, gt_nec, (1,0,0), alpha_cls, "GT • Necrose (1)")
        overlay(axes[r_gt,4], img2d, gt_enh, (0,0,1), alpha_cls, "GT • Enhancing (3)")
        overlay(axes[r_gt,5], img2d, gt_wt,  (1,1,0), alpha_comp,"GT • Whole Tumor (WT)")
        overlay(axes[r_gt,6], img2d, gt_tc,  (1,0,1), alpha_comp,"GT • Tumor Core (TC)")

        axes[r_pr,0].imshow(img2d, cmap="gray", origin="lower")
        axes[r_pr,0].set_title(f"PRED • {mod.upper()}", fontsize=8); axes[r_pr,0].axis("off")
        axes[r_pr,1].imshow(pr2d, cmap=cmap_mask, norm=norm_mask, origin="lower")
        axes[r_pr,1].set_title("PRED • Mask", fontsize=8); axes[r_pr,1].axis("off")
        overlay(axes[r_pr,2], img2d, pr_ede, (0,1,0), alpha_cls, "PRED • Edema (2)")
        overlay(axes[r_pr,3], img2d, pr_nec, (1,0,0), alpha_cls, "PRED • Necrose (1)")
        overlay(axes[r_pr,4], img2d, pr_enh, (0,0,1), alpha_cls, "PRED • Enhancing (3)")
        overlay(axes[r_pr,5], img2d, pr_wt,  (1,1,0), alpha_comp,"PRED • Whole Tumor (WT)")
        overlay(axes[r_pr,6], img2d, pr_tc,  (1,0,1), alpha_comp,"PRED • Tumor Core (TC)")

    plt.tight_layout(); plt.show()
    return cid, z, {"dice_c1":d1, "dice_c2":d2, "dice_ET":d3, "dice_WT":wt, "dice_TC":tc}


cid, z, dice_dict = plot_random_case_multimodal_gt_pred(test_ids, split_name="test", seed=None)

In [ ]:
# [1/6] Volume tumoral por caso

vol_rows = []
for cid in tqdm(test_ids, desc="Volume tumoral GT"):
    gt_path = find_file(case_dir("test", cid), "seg")
    gt = load_brats_seg(gt_path)
    spacing = nib.load(str(gt_path)).header.get_zooms()[:3]
    voxel_mm3 = float(np.prod(spacing))

    vol_rows.append({
        "id":         cid,
        "vol_WT_mm3": int(((gt==1)|(gt==2)|(gt==3)).sum()) * voxel_mm3,
        "vol_TC_mm3": int(((gt==1)|(gt==3)).sum())         * voxel_mm3,
        "vol_ET_mm3": int((gt==3).sum())                   * voxel_mm3,
    })

df_vol = pd.DataFrame(vol_rows)

df_err = (df.merge(df_hd95, on="id")
            .merge(lw_df,   on="id")
            .merge(df_vol,  on="id"))

df_err.describe().round(3)

In [ ]:
# [2/6] Métricas agregadas por quartil de tamanho tumoral (quartiles do volume WT)

q25, q50, q75 = df_err["vol_WT_mm3"].quantile([0.25, 0.50, 0.75]).values

def size_bin(v):
    if v < q25: return "Q1"
    if v < q50: return "Q2"
    if v < q75: return "Q3"
    return "Q4"

df_err["size_bin"] = df_err["vol_WT_mm3"].apply(size_bin)

agg_cols = ["dice_WT", "dice_TC", "dice_ET",
            "hd95_WT", "hd95_TC", "hd95_ET",
            "lwd_WT",  "lwd_TC",  "lwd_ET"]

df_by_size = (df_err.groupby("size_bin")[agg_cols].mean()
              .reindex(["Q1", "Q2", "Q3", "Q4"]).round(4))

print(f"Limiares (vol WT em mm³): Q1 < {q25:,.0f}  |  Q2 < {q50:,.0f}  |  Q3 < {q75:,.0f}  |  Q4 ≥ {q75:,.0f}")
print(f"Contagem por quartil: {df_err['size_bin'].value_counts().to_dict()}\n")

df_by_size

In [ ]:
# [3/6] Scatter: volume tumoral × Dice e × HD95 (por região)

regions = ["WT", "TC", "ET"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for j, reg in enumerate(regions):
    vol = df_err[f"vol_{reg}_mm3"].values
    dsc = df_err[f"dice_{reg}"].values
    hsc = df_err[f"hd95_{reg}"].values

    axes[0, j].scatter(vol, dsc, alpha=0.6, s=30)
    axes[0, j].set_xscale("log")
    axes[0, j].set_xlabel(f"Volume {reg} (mm³, log)")
    axes[0, j].set_ylabel("Dice")
    axes[0, j].set_title(f"{reg}: Volume × Dice")
    axes[0, j].grid(True, alpha=0.3)

    axes[1, j].scatter(vol, hsc, alpha=0.6, s=30, color="C1")
    axes[1, j].set_xscale("log")
    axes[1, j].set_xlabel(f"Volume {reg} (mm³, log)")
    axes[1, j].set_ylabel("HD95 (mm)")
    axes[1, j].set_title(f"{reg}: Volume × HD95")
    axes[1, j].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
# [3/6] Correlação de Spearman entre volume tumoral e métricas

regions = ["WT", "TC", "ET"]

corr_rows = []
for reg in regions:
    v = df_err[f"vol_{reg}_mm3"].values
    d = df_err[f"dice_{reg}"].values
    h = df_err[f"hd95_{reg}"].values

    rho_d, p_d = spearmanr(v, d)
    mask = ~np.isnan(h)
    if mask.sum() > 2:
        rho_h, p_h = spearmanr(v[mask], h[mask])
    else:
        rho_h, p_h = np.nan, np.nan

    corr_rows.append({
        "regiao": reg,
        "spearman_vol×dice": round(rho_d, 3),  "p_value_dice": round(p_d, 4),
        "spearman_vol×hd95": round(rho_h, 3),  "p_value_hd95": round(p_h, 4),
    })

print("Correlação de Spearman entre volume tumoral e métricas:")
pd.DataFrame(corr_rows)

In [ ]:
# [4/6] Ranking dos piores casos por Dice médio (WT/TC/ET)

K_WORST = 5

df_err["dice_mean"] = df_err[["dice_WT", "dice_TC", "dice_ET"]].mean(axis=1)
df_err["hd95_mean"] = df_err[["hd95_WT", "hd95_TC", "hd95_ET"]].mean(axis=1, skipna=True)

cols_show = ["id", "dice_WT", "dice_TC", "dice_ET",
             "hd95_WT", "hd95_TC", "hd95_ET",
             "vol_WT_mm3", "vol_ET_mm3", "size_bin"]

print(f"Top {K_WORST} piores casos por Dice médio:")
worst_dice = df_err.nsmallest(K_WORST, "dice_mean")[cols_show + ["dice_mean"]].round(3)
worst_ids = worst_dice["id"].tolist()

worst_dice

In [ ]:
# [4/6] Ranking dos piores casos por HD95 médio

print(f"Top {K_WORST} piores casos por HD95 médio:")
df_err.nlargest(K_WORST, "hd95_mean")[cols_show + ["hd95_mean"]].round(3)

In [ ]:
# [5/6] Visualização qualitativa dos piores casos

N_PLOT = min(1, len(worst_ids))

for cid in worst_ids[:N_PLOT]:
    plot_random_case_multimodal_gt_pred([cid], split_name="test", seed=0)

In [ ]:
# [6/6] Análise de componentes conexos: lesões perdidas (FN) vs (FP) por caso

CC_MIN_SIZE = 50

def count_components(gt_mask, pr_mask, min_size=CC_MIN_SIZE):
    structure = np.ones((3, 3, 3), dtype=int)
    gt_lab, n_gt = cc_label(gt_mask.astype(bool), structure=structure)
    pr_lab, n_pr = cc_label(pr_mask.astype(bool), structure=structure)

    n_gt_valid = sum(1 for g in range(1, n_gt + 1) if (gt_lab == g).sum() >= min_size)
    n_pr_valid = sum(1 for p in range(1, n_pr + 1) if (pr_lab == p).sum() >= min_size)

    matched_pred, n_missed = set(), 0
    for g in range(1, n_gt + 1):
        gt_l = (gt_lab == g)
        if gt_l.sum() < min_size:
            continue
        overlap = np.unique(pr_lab[gt_l]); overlap = overlap[overlap > 0]
        if len(overlap) == 0:
            n_missed += 1
        else:
            matched_pred.update(int(p) for p in overlap)

    n_spurious = sum(1 for p in range(1, n_pr + 1)
                     if p not in matched_pred and (pr_lab == p).sum() >= min_size)
    return n_gt_valid, n_pr_valid, n_missed, n_spurious


cc_rows = []
for cid in tqdm(test_ids, desc="Componentes conexos"):
    gt = load_brats_seg(find_file(case_dir("test", cid), "seg"))
    pr = load_arr(get_pred_path(cid)).astype(np.int16)

    nw = count_components((gt==1)|(gt==2)|(gt==3), (pr==1)|(pr==2)|(pr==3))
    nt = count_components((gt==1)|(gt==3),         (pr==1)|(pr==3))
    ne = count_components((gt==3),                  (pr==3))

    cc_rows.append({"id": cid,
        "WT_n_gt": nw[0], "WT_n_pr": nw[1], "WT_missed_FN": nw[2], "WT_spurious_FP": nw[3],
        "TC_n_gt": nt[0], "TC_n_pr": nt[1], "TC_missed_FN": nt[2], "TC_spurious_FP": nt[3],
        "ET_n_gt": ne[0], "ET_n_pr": ne[1], "ET_missed_FN": ne[2], "ET_spurious_FP": ne[3],
    })

df_cc = pd.DataFrame(cc_rows)

print("Totais agregados (todos os casos):")
totals = pd.DataFrame({
    "região":          ["WT", "TC", "ET"],
    "tot_lesões_GT":   [df_cc["WT_n_gt"].sum(),       df_cc["TC_n_gt"].sum(),       df_cc["ET_n_gt"].sum()],
    "tot_lesões_PR":   [df_cc["WT_n_pr"].sum(),       df_cc["TC_n_pr"].sum(),       df_cc["ET_n_pr"].sum()],
    "tot_perdidas_FN": [df_cc["WT_missed_FN"].sum(),  df_cc["TC_missed_FN"].sum(),  df_cc["ET_missed_FN"].sum()],
    "tot_espúrias_FP": [df_cc["WT_spurious_FP"].sum(),df_cc["TC_spurious_FP"].sum(),df_cc["ET_spurious_FP"].sum()],
})
totals["FN_rate_%"] = (100 * totals["tot_perdidas_FN"] / totals["tot_lesões_GT"].replace(0, np.nan)).round(2)
totals["FP_rate_%"] = (100 * totals["tot_espúrias_FP"] / totals["tot_lesões_PR"].replace(0, np.nan)).round(2)

totals

In [ ]:
# [6/6] Top 10 casos com mais erros de componente

print("Top 10 casos com mais erros de componente (FN+FP somados em todas as regiões):")

df_cc["err_total"] = df_cc[["WT_missed_FN","TC_missed_FN","ET_missed_FN",
                            "WT_spurious_FP","TC_spurious_FP","ET_spurious_FP"]].sum(axis=1)

df_cc.nlargest(10, "err_total")